# release16_rulebase_reversal_failed_fold_diagnostic_v1

Notebook-first root-cause diagnostic for the failed folds from `release16_rulebase_reversal_checkpoint_metric_replay_v1`.

- **Canonical baseline**: retained production frontier; promotion still requires fixed-split, `wf_2020`, `wf_2022`, `wf_2023_2024`, YTD alignment, dashboard replay evidence, and actor-retention governance.
- **Direct comparison baseline**: `release16_rulebase_reversal_checkpoint_metric_replay_v1`, which failed `wf_2020` and `wf_2023_2024` after the rulebase-reversal target contract replay.
- **Reference baseline**: `soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02` as the buy/sell timing reference with walk-forward drawdown under 20%.
- **Role**: `diagnose`; this card does not train, promote, or retain an actor.
- **Target window**: failed walk-forward folds `wf_2020` and `wf_2023_2024`; `wf_2022` remains a guard fold that already passed.
- **Changed layer**: `model_selection` / failure attribution / target revision routing.
- **External reference**: SOXLBacktest qlib buy/sell reference, official daily-path metric basis, and prior release16 checkpoint screens.
- **Hypothesis**: the rulebase reversal target increased participation enough to improve some return windows, but it either raised exposure during drawdown-sensitive periods (`wf_2020`) or still missed large rebound/upside days (`wf_2023_2024`).
- **Acceptance criteria**: identify date-level and month-level failure drivers, compare against immediate source overlay and SOXLBacktest reference, and produce a concrete next target-revision direction without claiming promotion.
- **Closest success case**: `release16_rulebase_reversal_checkpoint_metric_replay_v1` passed `wf_2022` with rel `71.71pp` and MaxDD `15.62%`.
- **Closest failure case**: `release16_failedfold_overlay_checkpoint_screen_v1` left `wf_2020` and `wf_2023_2024` below the 30pp relative-return target, then the rulebase-reversal replay moved risk/return but still failed both folds.
- **Why this is not a duplicate**: this card analyzes executed official daily paths after the rulebase-reversal replay and compares them against both the immediate source overlay and SOXLBacktest reference; it does not rerun the same checkpoint selection.


In [1]:

from __future__ import annotations

import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'AGENTS.md').exists():
    raise RuntimeError('Could not locate repo root')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

EXPERIMENT = 'release16_rulebase_reversal_failed_fold_diagnostic_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

SOURCE_REPLAY = 'release16_rulebase_reversal_checkpoint_metric_replay_v1'
SOURCE_OVERLAY = 'release16_failedfold_overlay_checkpoint_screen_v1'
CB99_SCREEN = 'release16_cb99_rebound_checkpoint_screen_v1'
READINESS_CARD = 'release16_rulebase_reversal_checkpoint_readiness_v1'
REFERENCE_CARD = 'soxlbacktest_qlib_walkforward_baseline_v1'
REFERENCE_CANDIDATE = 'wf_pass_ytd18_15_15_02'
FAILED_FOLDS = ['wf_2020', 'wf_2023_2024']
TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0
CREATED_AT = datetime.now(timezone.utc).isoformat()

BASE = ROOT / 'runtime' / 'model_selection' / 'card'

def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.floating, float)):
        value = float(value)
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        try:
            return str(value.relative_to(ROOT))
        except ValueError:
            return str(value)
    return value

def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')

def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))

def pct_return(values: pd.Series) -> float:
    clean = pd.to_numeric(values, errors='coerce').dropna()
    if len(clean) < 2:
        return 0.0
    start = float(clean.iloc[0])
    end = float(clean.iloc[-1])
    if not math.isfinite(start) or start == 0.0:
        return float('nan')
    return float((end / start - 1.0) * 100.0)

def max_drawdown_pct(values: pd.Series) -> float:
    clean = pd.to_numeric(values, errors='coerce').dropna()
    if clean.empty:
        return 0.0
    peak = clean.cummax()
    dd = clean / peak - 1.0
    return float(-dd.min() * 100.0)

source_summary = load_json(BASE / SOURCE_REPLAY / 'summary.json')
readiness_summary = load_json(BASE / READINESS_CARD / 'summary.json')
source_selection = pd.read_csv(BASE / SOURCE_REPLAY / 'checkpoint_selection_by_fold.csv')
source_candidates = pd.read_csv(BASE / SOURCE_REPLAY / 'checkpoint_candidate_metrics.csv')
reference_daily = pd.read_csv(BASE / REFERENCE_CARD / 'candidate_daily.csv')
reference_metrics = pd.read_csv(BASE / REFERENCE_CARD / 'candidate_metrics.csv')
print('source decision', source_summary['decision'])
display(source_selection)


source decision release16_rulebase_reversal_checkpoint_metric_replay_partial_pass_no_promotion


,fold,status,selected_candidate,selected_kind,selected_iter,selected_target_gate,selected_drift_gate,selected_candidate_gate,selected_loss,selected_relative_return,...,candidate_count,reference_candidate,reference_return_pct,reference_relative_return_pp,reference_max_dd_pct,reference_avg_position,reference_trade_count,selected_vs_reference_return_delta_pct,selected_vs_reference_relative_delta_pp,selected_vs_reference_max_dd_delta_pct
0,wf_2020,evaluated,bc_initialization,bc_initialization,0,False,True,False,0.877307,5.082695,...,4,wf_pass_ytd18_15_15_02,29.156603,-30.618705,12.785207,0.171011,2,34.019662,35.701400,8.149405
1,wf_2022,evaluated,bc_initialization,bc_initialization,0,True,True,True,0.000000,71.714980,...,4,wf_pass_ytd18_15_15_02,-16.044551,70.543521,18.514585,0.102318,6,1.171459,1.171459,-2.897542
2,wf_2023_2024,evaluated,bc_initialization,bc_initialization,0,False,True,False,0.614049,11.578539,...,4,wf_pass_ytd18_15_15_02,4.505779,7.109208,1.061804,0.013221,5,6.716125,4.469331,10.688224


In [2]:

def selected_paths(card_name: str) -> dict[str, Path]:
    frame = pd.read_csv(BASE / card_name / 'checkpoint_selection_by_fold.csv')
    return {str(row['fold']): ROOT / str(row['selected_daily_path']) for _, row in frame.iterrows()}

PATHS = {
    'new': selected_paths(SOURCE_REPLAY),
    'source_overlay': selected_paths(SOURCE_OVERLAY),
    'cb99': selected_paths(CB99_SCREEN),
}

def prep_model_daily(path: Path, prefix: str) -> pd.DataFrame:
    frame = pd.read_csv(path)
    frame['datetime'] = pd.to_datetime(frame['datetime']).dt.normalize()
    cols = ['datetime', 'strategy_nav', 'buyhold_nav', 'position', 'price', 'action']
    out = frame[cols].copy().sort_values('datetime')
    out[f'{prefix}_daily_ret'] = out['strategy_nav'].astype(float).pct_change().fillna(0.0)
    out[f'{prefix}_drawdown_pct'] = (out['strategy_nav'].astype(float) / out['strategy_nav'].astype(float).cummax() - 1.0) * 100.0
    return out.rename(columns={col: f'{prefix}_{col}' for col in cols if col != 'datetime'})

def prep_reference_daily(fold: str) -> pd.DataFrame:
    frame = reference_daily[(reference_daily['candidate'].eq(REFERENCE_CANDIDATE)) & (reference_daily['window'].eq(fold))].copy()
    frame['datetime'] = pd.to_datetime(frame['date']).dt.normalize()
    cols = ['datetime', 'strategy_nav', 'buyhold_nav', 'position', 'price', 'action']
    out = frame[cols].copy().sort_values('datetime')
    out['reference_daily_ret'] = out['strategy_nav'].astype(float).pct_change().fillna(0.0)
    out['reference_drawdown_pct'] = (out['strategy_nav'].astype(float) / out['strategy_nav'].astype(float).cummax() - 1.0) * 100.0
    return out.rename(columns={col: f'reference_{col}' for col in cols if col != 'datetime'})

def merged_daily(fold: str) -> pd.DataFrame:
    work = prep_model_daily(PATHS['new'][fold], 'new')
    for label in ['source_overlay', 'cb99']:
        work = work.merge(prep_model_daily(PATHS[label][fold], label), on='datetime', how='inner')
    work = work.merge(prep_reference_daily(fold), on='datetime', how='inner')
    work['price_return'] = pd.to_numeric(work['new_price'], errors='coerce').pct_change().fillna(0.0)
    work['fold'] = fold
    work['month'] = work['datetime'].dt.to_period('M').astype(str)
    work['new_minus_source_daily_ret'] = work['new_daily_ret'] - work['source_overlay_daily_ret']
    work['new_minus_reference_daily_ret'] = work['new_daily_ret'] - work['reference_daily_ret']
    work['new_position_minus_source'] = work['new_position'] - work['source_overlay_position']
    work['new_position_minus_reference'] = work['new_position'] - work['reference_position']
    work['overexposure_down_score'] = np.where(
        work['price_return'] < 0.0,
        work['new_position_minus_source'].clip(lower=0.0) * (-work['price_return']),
        0.0,
    )
    work['underparticipation_up_score'] = np.where(
        work['price_return'] > 0.0,
        (0.35 - work['new_position']).clip(lower=0.0) * work['price_return'],
        0.0,
    )
    return work

failed_daily = pd.concat([merged_daily(fold) for fold in FAILED_FOLDS], ignore_index=True)
failed_daily.to_csv(CARD / 'failed_fold_daily_comparison.csv', index=False)
print('daily rows', failed_daily.shape)
display(failed_daily.head())


daily rows (503, 38)


,datetime,new_strategy_nav,new_buyhold_nav,new_position,new_price,new_action,new_daily_ret,new_drawdown_pct,source_overlay_strategy_nav,source_overlay_buyhold_nav,...,reference_drawdown_pct,price_return,fold,month,new_minus_source_daily_ret,new_minus_reference_daily_ret,new_position_minus_source,new_position_minus_reference,overexposure_down_score,underparticipation_up_score
0,2020-01-02,1000000.0,1000000.000000,0.250000,19.464001,HOLD,0.0,0.0,1000000.0,1000000.000000,...,0.0,0.000000,wf_2020,2020-01,0.0,0.0,0.0,0.250000,0.0,0.000000
1,2020-01-03,1000000.0,945985.699589,0.233747,18.412666,HOLD,0.0,0.0,1000000.0,945985.699589,...,0.0,-0.054014,wf_2020,2020-01,0.0,0.0,0.0,0.233747,0.0,0.000000
2,2020-01-06,1000000.0,915981.609334,0.203625,17.828667,HOLD,0.0,0.0,1000000.0,915981.609334,...,0.0,-0.031717,wf_2020,2020-01,0.0,0.0,0.0,0.203625,0.0,0.000000
3,2020-01-07,1000000.0,963214.062639,0.218200,18.747999,HOLD,0.0,0.0,1000000.0,963214.062639,...,0.0,0.051565,wf_2020,2020-01,0.0,0.0,0.0,0.218200,0.0,0.006796
4,2020-01-08,1000000.0,965337.683206,0.207931,18.789333,HOLD,0.0,0.0,1000000.0,965337.683206,...,0.0,0.002205,wf_2020,2020-01,0.0,0.0,0.0,0.207931,0.0,0.000313


In [3]:

def maxdd_segment(frame: pd.DataFrame, nav_col: str) -> dict[str, Any]:
    nav = pd.to_numeric(frame[nav_col], errors='coerce').reset_index(drop=True)
    peak = nav.cummax()
    drawdown = nav / peak - 1.0
    trough_idx = int(drawdown.idxmin())
    peak_idx = int(nav.loc[:trough_idx].idxmax())
    segment = frame.iloc[peak_idx:trough_idx + 1].copy()
    prefix = nav_col.removesuffix('_strategy_nav')
    return {
        'path': prefix,
        'peak_date': str(segment['datetime'].iloc[0].date()),
        'trough_date': str(segment['datetime'].iloc[-1].date()),
        'segment_rows': int(len(segment)),
        'max_dd_pct': float(-drawdown.iloc[trough_idx] * 100.0),
        'segment_price_return_pct': pct_return(segment['new_price']),
        'segment_new_return_pct': pct_return(segment['new_strategy_nav']),
        'segment_source_overlay_return_pct': pct_return(segment['source_overlay_strategy_nav']),
        'segment_reference_return_pct': pct_return(segment['reference_strategy_nav']),
        'avg_new_position': float(segment['new_position'].mean()),
        'avg_source_overlay_position': float(segment['source_overlay_position'].mean()),
        'avg_reference_position': float(segment['reference_position'].mean()),
    }

segment_rows = []
for fold, group in failed_daily.groupby('fold', sort=False):
    for col in ['new_strategy_nav', 'source_overlay_strategy_nav', 'reference_strategy_nav', 'cb99_strategy_nav']:
        row = maxdd_segment(group.reset_index(drop=True), col)
        row.insert(0, 'fold') if hasattr(row, 'insert') else None
        row = {'fold': fold, **row}
        segment_rows.append(row)
segments = pd.DataFrame(segment_rows)
segments.to_csv(CARD / 'failed_fold_drawdown_segments.csv', index=False)
display(segments)


,fold,path,peak_date,trough_date,segment_rows,max_dd_pct,segment_price_return_pct,segment_new_return_pct,segment_source_overlay_return_pct,segment_reference_return_pct,avg_new_position,avg_source_overlay_position,avg_reference_position
0,wf_2020,new,2020-09-02,2020-09-08,4,20.934612,-30.847297,-20.934612,-16.652151,-8.240690,0.460179,0.191264,0.226500
1,wf_2020,source_overlay,2020-09-02,2020-09-08,4,16.652151,-30.847297,-20.934612,-16.652151,-8.240690,0.460179,0.191264,0.226500
2,wf_2020,reference,2020-03-04,2020-03-20,13,12.785207,-72.450785,0.000000,0.000000,-12.785207,0.296961,0.160917,0.159100
3,wf_2020,cb99,2020-09-02,2020-09-08,4,14.961639,-30.847297,-20.934612,-16.652151,-8.240690,0.460179,0.191264,0.226500
4,wf_2023_2024,new,2024-07-10,2024-12-19,115,11.750028,-61.656217,-11.750028,-5.376170,-0.206407,0.116344,0.096642,0.019175
5,wf_2023_2024,source_overlay,2024-09-26,2024-12-19,60,5.546722,-33.669614,-7.533155,-5.546722,-0.900124,0.124861,0.106567,0.021662
6,wf_2023_2024,reference,2024-08-21,2024-09-06,12,1.061804,-37.790560,0.000000,0.000000,-1.061804,0.096211,0.096431,0.027000
7,wf_2023_2024,cb99,2024-09-26,2024-12-19,60,5.618566,-33.669614,-7.533155,-5.546722,-0.900124,0.124861,0.106567,0.021662


In [4]:

monthly_rows = []
for (fold, month), group in failed_daily.groupby(['fold', 'month'], sort=True):
    row = {
        'fold': fold,
        'month': month,
        'rows': int(len(group)),
        'price_return_pct': pct_return(group['new_price']),
        'new_return_pct': pct_return(group['new_strategy_nav']),
        'source_overlay_return_pct': pct_return(group['source_overlay_strategy_nav']),
        'cb99_return_pct': pct_return(group['cb99_strategy_nav']),
        'reference_return_pct': pct_return(group['reference_strategy_nav']),
        'new_avg_position': float(group['new_position'].mean()),
        'source_overlay_avg_position': float(group['source_overlay_position'].mean()),
        'reference_avg_position': float(group['reference_position'].mean()),
    }
    row['new_minus_source_return_pp'] = row['new_return_pct'] - row['source_overlay_return_pct']
    row['new_minus_reference_return_pp'] = row['new_return_pct'] - row['reference_return_pct']
    row['new_position_minus_source'] = row['new_avg_position'] - row['source_overlay_avg_position']
    monthly_rows.append(row)
monthly = pd.DataFrame(monthly_rows)
monthly.to_csv(CARD / 'failed_fold_monthly_attribution.csv', index=False)

gap_rows = []
for fold, group in failed_daily.groupby('fold', sort=False):
    for label, score_col, ascending in [
        ('overexposure_down', 'overexposure_down_score', False),
        ('underparticipation_up', 'underparticipation_up_score', False),
        ('new_worse_than_source_daily_return', 'new_minus_source_daily_ret', True),
        ('new_worse_than_reference_daily_return', 'new_minus_reference_daily_ret', True),
    ]:
        cols = [
            'fold', 'datetime', 'price_return', 'new_position', 'source_overlay_position', 'reference_position',
            'new_daily_ret', 'source_overlay_daily_ret', 'reference_daily_ret', 'new_minus_source_daily_ret',
            'new_minus_reference_daily_ret', 'overexposure_down_score', 'underparticipation_up_score',
        ]
        top = group.sort_values(score_col, ascending=ascending).head(12).copy()
        top['gap_type'] = label
        gap_rows.append(top[['gap_type', *cols]])
top_gap_days = pd.concat(gap_rows, ignore_index=True)
top_gap_days.to_csv(CARD / 'failed_fold_top_gap_days.csv', index=False)

display(monthly.sort_values(['fold', 'new_minus_source_return_pp']).groupby('fold').head(5))
display(top_gap_days.head(12))


,fold,month,rows,price_return_pct,new_return_pct,source_overlay_return_pct,cb99_return_pct,reference_return_pct,new_avg_position,source_overlay_avg_position,reference_avg_position,new_minus_source_return_pp,new_minus_reference_return_pp,new_position_minus_source
9,wf_2020,2020-10,22,-7.422562,-4.710436,3.149864,-1.370920,-1.794425,0.099816,0.097822,0.246600,-7.860299,-2.916011,0.001993
5,wf_2020,2020-06,22,23.659654,0.683143,5.834458,6.426099,3.616824,0.486804,0.411648,0.157509,-5.151315,-2.933681,0.075156
8,wf_2020,2020-09,21,-10.260865,-4.600098,-2.567446,-6.192436,-2.572043,0.505204,0.303784,0.223214,-2.032653,-2.028056,0.201421
6,wf_2020,2020-07,22,25.601641,8.917294,9.794691,6.419677,4.509825,0.368892,0.254163,0.190527,-0.877396,4.407469,0.114729
11,wf_2020,2020-12,21,7.967063,0.976552,1.710853,1.702258,2.679917,0.099816,0.097822,0.313433,-0.734301,-1.703364,0.001993
14,wf_2023_2024,2024-03,20,-3.724401,-0.799236,0.000000,0.000000,0.000000,0.428507,0.118908,0.000000,-0.799236,-0.799236,0.309600
19,wf_2023_2024,2024-08,22,10.828574,0.764556,1.553155,0.796650,0.641627,0.069418,0.067741,0.021900,-0.788600,0.122929,0.001676
15,wf_2023_2024,2024-04,22,-20.441761,-2.207640,-1.594125,-0.538192,4.570119,0.200294,0.103775,0.039177,-0.613516,-6.777760,0.096519
21,wf_2023_2024,2024-10,23,-11.055281,-1.906510,-1.811814,-1.919708,-0.253163,0.100000,0.100000,0.022248,-0.094695,-1.653346,0.000000
20,wf_2023_2024,2024-09,20,21.982044,0.452942,0.463240,0.457061,0.448577,0.251383,0.140360,0.024345,-0.010298,0.004365,0.111023


,gap_type,fold,datetime,price_return,new_position,source_overlay_position,reference_position,new_daily_ret,source_overlay_daily_ret,reference_daily_ret,new_minus_source_daily_ret,new_minus_reference_daily_ret,overexposure_down_score,underparticipation_up_score
0,overexposure_down,wf_2020,2020-03-16,-0.385930,0.452838,0.099748,0.1591,0.000000,0.000000,-0.041328,0.000000,0.041328,0.136268,0.0
1,overexposure_down,wf_2020,2020-03-12,-0.304146,0.450000,0.098775,0.1591,0.000000,0.000000,-0.036262,0.000000,0.036262,0.106824,0.0
2,overexposure_down,wf_2020,2020-09-03,-0.170662,0.450000,0.104507,0.2265,-0.125685,-0.078880,-0.045591,-0.046806,-0.080094,0.058963,0.0
3,overexposure_down,wf_2020,2020-03-18,-0.258921,0.410879,0.285054,0.1591,0.000000,0.000000,-0.019394,0.000000,0.019394,0.032579,0.0
4,overexposure_down,wf_2020,2020-03-20,-0.085514,0.650000,0.304718,0.1591,0.000000,0.000000,-0.005185,0.000000,0.005185,0.029526,0.0
5,overexposure_down,wf_2020,2020-05-01,-0.151973,0.270599,0.098742,0.1336,-0.034939,-0.017183,-0.020432,-0.017756,-0.014507,0.026118,0.0
6,overexposure_down,wf_2020,2020-04-30,-0.105123,0.282799,0.104302,0.1336,-0.026297,-0.013108,-0.015548,-0.013189,-0.010749,0.018764,0.0
7,overexposure_down,wf_2020,2020-09-23,-0.074453,0.702055,0.450776,0.2196,-0.033001,-0.033054,-0.016132,0.000053,-0.016869,0.018709,0.0
8,overexposure_down,wf_2020,2020-05-12,-0.084776,0.273805,0.103145,0.1336,-0.022493,-0.018992,-0.011965,-0.003500,-0.010527,0.014468,0.0
9,overexposure_down,wf_2020,2020-05-21,-0.077695,0.279543,0.105827,0.1336,-0.009326,-0.008476,-0.011890,-0.000850,0.002564,0.013497,0.0


In [5]:

overlay_rows = pd.read_csv(BASE / READINESS_CARD / 'rulebase_reversal_overlay_rows.csv')
target_contract = pd.read_csv(BASE / READINESS_CARD / 'rulebase_reversal_bc_target_contract.csv')
for frame in [overlay_rows, target_contract]:
    frame['datetime'] = pd.to_datetime(frame['datetime']).dt.normalize()
    if 'fold_family' not in frame.columns:
        frame['fold_family'] = frame['fold'].astype(str).str.replace('_train', '', regex=False)

def bool_count(frame: pd.DataFrame, col: str) -> int:
    if col not in frame.columns:
        return 0
    series = frame[col]
    if series.dtype == object:
        return int(series.astype(str).str.lower().isin({'true', '1', 'yes', 'y', 't'}).sum())
    return int(series.fillna(False).astype(bool).sum())

target_rows = []
for fold, group in target_contract.groupby('fold_family', sort=True):
    overlay_group = overlay_rows[overlay_rows['fold_family'].eq(fold)]
    target_rows.append({
        'fold_family': fold,
        'contract_rows': int(len(group)),
        'rulebase_overlay_rows': int(len(overlay_group)),
        'overlay_share_pct': float(len(overlay_group) / max(len(group), 1) * 100.0),
        'mean_rulebase_target_lift': float(pd.to_numeric(overlay_group.get('rulebase_reversal_target_lift', pd.Series(dtype=float)), errors='coerce').mean()) if not overlay_group.empty else 0.0,
        'mean_consumed_target_delta': float(pd.to_numeric(overlay_group.get('rulebase_consumed_target_delta', pd.Series(dtype=float)), errors='coerce').mean()) if not overlay_group.empty else 0.0,
        'median_consumed_target_delta': float(pd.to_numeric(overlay_group.get('rulebase_consumed_target_delta', pd.Series(dtype=float)), errors='coerce').median()) if not overlay_group.empty else 0.0,
        'release_gate_rows': bool_count(group, 'release_gate_state'),
        'downtrend_rows': bool_count(group, 'is_downtrend_state'),
        'rebound_rows': bool_count(group, 'is_rebound_cluster'),
        'factor_rebound_rows': bool_count(group, 'factor_rebound_state'),
        'min_overlay_date': str(overlay_group['datetime'].min().date()) if not overlay_group.empty else '',
        'max_overlay_date': str(overlay_group['datetime'].max().date()) if not overlay_group.empty else '',
    })
target_pressure = pd.DataFrame(target_rows)
target_pressure.to_csv(CARD / 'target_pressure_summary.csv', index=False)

action_rows = []
selection_by_fold = source_selection.set_index('fold').to_dict('index')
segment_by_fold = segments[segments['path'].eq('new')].set_index('fold').to_dict('index')
for fold in FAILED_FOLDS:
    sel = selection_by_fold[fold]
    seg = segment_by_fold[fold]
    if fold == 'wf_2020':
        blocker = 'drawdown_overexposure_after_rulebase_lift'
        action = 'build_drawdown_capped_rulebase_reversal_target_for_wf2020'
        rationale = (
            f"new MaxDD {sel['selected_max_dd']:.2f}% exceeds 20%; max drawdown segment "
            f"{seg['peak_date']}->{seg['trough_date']} had avg new position {seg['avg_new_position']:.3f} "
            f"vs source {seg['avg_source_overlay_position']:.3f}."
        )
        target_hint = 'cap reversal lifts when recent drawdown/volshock exposure would exceed the source overlay; keep rebound gains without September-style overexposure.'
    else:
        blocker = 'upside_underparticipation_after_safe_drawdown'
        action = 'build_high_participation_rebound_target_for_wf2024_with_drawdown_guard'
        rationale = (
            f"new MaxDD {sel['selected_max_dd']:.2f}% is below 20%, but relative return {sel['selected_relative_return']:.2f}pp "
            f"is {TARGET_RELATIVE_RETURN_PP - sel['selected_relative_return']:.2f}pp below target."
        )
        target_hint = 'add short-horizon upside/rebound participation targets for large positive SOXL days while preserving the existing drawdown guard.'
    action_rows.append({
        'fold': fold,
        'blocker': blocker,
        'recommended_next_action': action,
        'rationale': rationale,
        'target_revision_hint': target_hint,
        'selected_relative_return_pp': float(sel['selected_relative_return']),
        'selected_max_dd_pct': float(sel['selected_max_dd']),
        'relative_gap_to_30pp': float(TARGET_RELATIVE_RETURN_PP - sel['selected_relative_return']),
        'drawdown_gap_to_20pct': float(max(0.0, sel['selected_max_dd'] - TARGET_MAX_DD_PCT)),
    })
revision_direction = pd.DataFrame(action_rows)
revision_direction.to_csv(CARD / 'revision_direction_matrix.csv', index=False)

display(target_pressure)
display(revision_direction)


,fold_family,contract_rows,rulebase_overlay_rows,overlay_share_pct,mean_rulebase_target_lift,mean_consumed_target_delta,median_consumed_target_delta,release_gate_rows,downtrend_rows,rebound_rows,factor_rebound_rows,min_overlay_date,max_overlay_date
0,wf_2020,2470,424,17.165992,0.145471,0.144785,0.122600,42,781,42,0,2011-04-21,2019-04-29
1,wf_2022,505,0,0.000000,0.000000,0.000000,0.000000,9,234,13,9,,
2,wf_2023_2024,756,145,19.179894,0.310923,0.301670,0.316239,11,478,14,11,2020-01-14,2022-01-03


,fold,blocker,recommended_next_action,rationale,target_revision_hint,selected_relative_return_pp,selected_max_dd_pct,relative_gap_to_30pp,drawdown_gap_to_20pct
0,wf_2020,drawdown_overexposure_after_rulebase_lift,build_drawdown_capped_rulebase_reversal_target...,new MaxDD 20.93% exceeds 20%; max drawdown seg...,cap reversal lifts when recent drawdown/volsho...,5.082695,20.934612,24.917305,0.934612
1,wf_2023_2024,upside_underparticipation_after_safe_drawdown,build_high_participation_rebound_target_for_wf...,"new MaxDD 11.75% is below 20%, but relative re...",add short-horizon upside/rebound participation...,11.578539,11.750028,18.421461,0.000000


In [6]:

metrics = source_summary.get('metrics', [])
failed_selection = source_selection[source_selection['fold'].isin(FAILED_FOLDS)].copy()
wf2020_segment = segments[(segments['fold'].eq('wf_2020')) & (segments['path'].eq('new'))].iloc[0].to_dict()
wf2024_segment = segments[(segments['fold'].eq('wf_2023_2024')) & (segments['path'].eq('new'))].iloc[0].to_dict()
wf2020_row = failed_selection[failed_selection['fold'].eq('wf_2020')].iloc[0].to_dict()
wf2024_row = failed_selection[failed_selection['fold'].eq('wf_2023_2024')].iloc[0].to_dict()

conclusion_lines = [
    f"wf_2020 failure is now drawdown-dominated: rel {wf2020_row['selected_relative_return']:.2f}pp and MaxDD {wf2020_row['selected_max_dd']:.2f}%.",
    f"wf_2020 max drawdown segment {wf2020_segment['peak_date']}->{wf2020_segment['trough_date']} had price return {wf2020_segment['segment_price_return_pct']:.2f}% and avg new position {wf2020_segment['avg_new_position']:.3f} vs source overlay {wf2020_segment['avg_source_overlay_position']:.3f}; the rulebase lift crossed the 20% drawdown ceiling.",
    f"wf_2023_2024 failure is return-dominated: rel {wf2024_row['selected_relative_return']:.2f}pp leaves a {TARGET_RELATIVE_RETURN_PP - wf2024_row['selected_relative_return']:.2f}pp gap while MaxDD {wf2024_row['selected_max_dd']:.2f}% stays below 20%.",
    "PPO continuation did not solve either failed fold; selected candidates remain BC initialization, so the next useful lever is target/objective revision rather than longer PPO from this target.",
    "Do not promote or retain any actor from this diagnostic.",
]
next_mechanism = 'build_drawdown_capped_rulebase_reversal_and_wf2024_upside_participation_target_then_rerun_checkpoint_metric_replay'
selection = {
    'experiment': EXPERIMENT,
    'created_at': CREATED_AT,
    'decision': 'rulebase_reversal_failed_fold_diagnostic_requires_target_revision_no_promotion',
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': False,
    'target_revision_ready': False,
    'next_training_allowed_after_target_revision': False,
    'primary_blocker': 'walkforward_relative_return_gap',
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': SOURCE_REPLAY,
    'source_replay': SOURCE_REPLAY,
    'source_readiness_card': READINESS_CARD,
    'reference_baseline': f'{REFERENCE_CARD}:{REFERENCE_CANDIDATE}',
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'failure_attribution', 'target_revision_routing'],
    'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'failed_folds': FAILED_FOLDS,
    'metrics': metrics,
    'wf2020_drawdown_segment': wf2020_segment,
    'wf2023_2024_drawdown_segment': wf2024_segment,
    'revision_direction': revision_direction.to_dict('records'),
    'conclusion': conclusion_lines,
    'sidecars': [
        'failed_fold_daily_comparison.csv',
        'failed_fold_drawdown_segments.csv',
        'failed_fold_monthly_attribution.csv',
        'failed_fold_top_gap_days.csv',
        'target_pressure_summary.csv',
        'revision_direction_matrix.csv',
        'selection.json',
        'summary.json',
        'conclusion.md',
    ],
}
summary = {**selection}
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)

lines = [
    '# Release16 Rulebase Reversal Failed-Fold Diagnostic',
    '',
    f"Decision: `{selection['decision']}`.",
    '',
]
lines.extend([f'- {line}' for line in conclusion_lines])
lines.extend([
    '',
    f"Next action: `{next_mechanism}`.",
    '',
    'Artifacts: `failed_fold_daily_comparison.csv`, `failed_fold_drawdown_segments.csv`, `failed_fold_monthly_attribution.csv`, `failed_fold_top_gap_days.csv`, `target_pressure_summary.csv`, `revision_direction_matrix.csv`, `selection.json`, `summary.json`.',
])
conclusion_md = '\n'.join(lines).rstrip() + '\n'
(CARD / 'conclusion.md').write_text(conclusion_md, encoding='utf-8')
display(Markdown(conclusion_md))
print(json.dumps(json_safe(selection), indent=2, allow_nan=False)[:5000])


# Release16 Rulebase Reversal Failed-Fold Diagnostic

Decision: `rulebase_reversal_failed_fold_diagnostic_requires_target_revision_no_promotion`.

- wf_2020 failure is now drawdown-dominated: rel 5.08pp and MaxDD 20.93%.
- wf_2020 max drawdown segment 2020-09-02->2020-09-08 had price return -30.85% and avg new position 0.460 vs source overlay 0.191; the rulebase lift crossed the 20% drawdown ceiling.
- wf_2023_2024 failure is return-dominated: rel 11.58pp leaves a 18.42pp gap while MaxDD 11.75% stays below 20%.
- PPO continuation did not solve either failed fold; selected candidates remain BC initialization, so the next useful lever is target/objective revision rather than longer PPO from this target.
- Do not promote or retain any actor from this diagnostic.

Next action: `build_drawdown_capped_rulebase_reversal_and_wf2024_upside_participation_target_then_rerun_checkpoint_metric_replay`.

Artifacts: `failed_fold_daily_comparison.csv`, `failed_fold_drawdown_segments.csv`, `failed_fold_monthly_attribution.csv`, `failed_fold_top_gap_days.csv`, `target_pressure_summary.csv`, `revision_direction_matrix.csv`, `selection.json`, `summary.json`.


{
  "experiment": "release16_rulebase_reversal_failed_fold_diagnostic_v1",
  "created_at": "2026-06-14T16:50:53.836753+00:00",
  "decision": "rulebase_reversal_failed_fold_diagnostic_requires_target_revision_no_promotion",
  "promotion_allowed": false,
  "actor_retention_allowed": false,
  "training_allowed_from_this_card": false,
  "target_revision_ready": false,
  "next_training_allowed_after_target_revision": false,
  "primary_blocker": "walkforward_relative_return_gap",
  "next_mechanism": "build_drawdown_capped_rulebase_reversal_and_wf2024_upside_participation_target_then_rerun_checkpoint_metric_replay",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "release16_rulebase_reversal_checkpoint_metric_replay_v1",
  "source_replay": "release16_rulebase_reversal_checkpoint_metric_replay_v1",
  "source_readiness_card": "release16_rulebase_reversal_checkpoint_readiness_v1",
  "reference_baseline": "soxlbacktest_qlib_walkforward_baseline_v1: